In [1]:
import os, platform, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchxrayvision as xrv
import cv2
from dataclasses import dataclass
from typing import Dict
from torch.utils.data import Dataset, DataLoader
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import label_binarize
from tqdm import tqdm

d:\Anaconda\envs\TFM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
@dataclass
class CFG:
    DATA_ROOT      = 'E:/TFM/Dataset_desolapado'
    OUT_DIR        = 'E:/TFM/Nuevos_modelos/Outputs_knn_baseline'
    IMG_SIZE       = 224
    MAXVAL         = 65535.0
    XRV_WEIGHTS    = 'densenet121-res224-all'
    EMB_DIM        = 128
    K_NEIGHBORS    = 1
    N_REPEATS      = 100
    SEED           = 42
    DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg    = CFG()
MAXVAL = cfg.MAXVAL
device = torch.device(cfg.DEVICE)
os.makedirs(cfg.OUT_DIR, exist_ok=True)
print('Device:', device)

Device: cuda


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def list_images_in_folder(root):
    paths, ys, class_names = [], [], []
    classes = sorted(os.listdir(root))
    for i, cls in enumerate(classes):
        cls_dir = os.path.join(root, cls)
        if not os.path.isdir(cls_dir): continue
        class_names.append(cls)
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                paths.append(os.path.join(cls_dir, fname))
                ys.append(i)
    return paths, ys, class_names

In [4]:
class XRayFolderDataset(Dataset):
    def __init__(self, paths, ys, class_names, base_preproc=None,
                 cache_in_ram=True, cache_after_preproc=True):
        self.paths   = paths
        self.ys      = np.array(ys, dtype=np.int64)
        self.classes = class_names
        self.base_preproc = base_preproc
        self.cache_after_preproc = cache_after_preproc
        self.cache_in_ram = cache_in_ram
        self._cache: Dict[str, np.ndarray] = {}

    def __len__(self): return len(self.paths)

    def _load_raw(self, path):
        img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_ANYDEPTH)
        if img is None: raise RuntimeError(f'No pude leer {path}')
        img = img.astype(np.float32)
        img = xrv.datasets.normalize(img, MAXVAL)
        return img[None, ...]

    def _load_img(self, path):
        if self.cache_in_ram and path in self._cache:
            return self._cache[path].copy()
        img = self._load_raw(path)
        if self.base_preproc is not None and self.cache_after_preproc:
            img = self.base_preproc(img)
        if self.cache_in_ram:
            self._cache[path] = img.copy()
        return img.copy()

    def __getitem__(self, idx):
        img = self._load_img(self.paths[idx])
        return torch.from_numpy(img).float(), torch.tensor(int(self.ys[idx]), dtype=torch.long)

In [5]:
class XRVEncoder(nn.Module):
    def __init__(self, weights, emb_dim=128, freeze=True):
        super().__init__()
        self.backbone = xrv.models.DenseNet(weights=weights)
        self.proj = nn.Linear(1024, emb_dim)
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        feats = self.backbone.features2(x)
        return F.normalize(self.proj(feats), dim=1)

In [6]:
def extract_embeddings(ds, model, device, batch_size=32):
    model.eval()
    all_embs, all_labels = [], []
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc='Extrayendo embeddings', leave=False):
            imgs = imgs.to(device)
            embs = model(imgs)
            all_embs.append(embs.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_embs), np.concatenate(all_labels)

In [ ]:
def run_knn_baseline(k_shot=5, n_repeats=100):
    set_seed(cfg.SEED)
    rng = np.random.default_rng(cfg.SEED)

    base_preproc = T.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(cfg.IMG_SIZE),
    ])

    te_paths, te_y, test_cn = list_images_in_folder(os.path.join(cfg.DATA_ROOT, 'test'))
    test_ds = XRayFolderDataset(te_paths, te_y, test_cn, base_preproc=base_preproc)
    n_classes = len(test_cn)

    print(f'Test: {len(test_ds)} imgs, {n_classes} clases')
    print(f'K-shot={k_shot}, n_repeats={n_repeats}')

    model = XRVEncoder(cfg.XRV_WEIGHTS, emb_dim=cfg.EMB_DIM, freeze=True).to(device)

    print('Extrayendo embeddings de test...')
    te_embs, te_labels = extract_embeddings(test_ds, model, device)

    class_indices = {c: np.where(te_labels == c)[0] for c in range(n_classes)}

    accs, f1s, aucs = [], [], []

    for rep in tqdm(range(n_repeats), desc='Repeticiones'):
        support_idx, query_idx = [], []
        for c in range(n_classes):
            idx = class_indices[c]
            chosen = rng.choice(idx, size=k_shot, replace=len(idx) < k_shot)
            support_idx.extend(chosen.tolist())
            query_idx.extend([i for i in idx if i not in chosen])

        support_idx = np.array(support_idx)
        query_idx   = np.array(query_idx)

        S_embs  = te_embs[support_idx]
        S_labels = te_labels[support_idx]
        Q_embs  = te_embs[query_idx]
        Q_labels = te_labels[query_idx]

        knn = KNeighborsClassifier(n_neighbors=min(k_shot, len(S_embs)), metric='cosine')
        knn.fit(S_embs, S_labels)

        preds = knn.predict(Q_embs)
        probs = knn.predict_proba(Q_embs)

        acc = accuracy_score(Q_labels, preds)
        f1  = f1_score(Q_labels, preds, average='macro', zero_division=0)

        y_bin = label_binarize(Q_labels, classes=range(n_classes))
        if y_bin.shape[1] == 1:
            y_bin = np.hstack([1 - y_bin, y_bin])
        try:
            auc_score = roc_auc_score(y_bin, probs, average='macro', multi_class='ovr')
        except:
            auc_score = float('nan')

        accs.append(acc)
        f1s.append(f1)
        aucs.append(auc_score)

    accs = np.array(accs)
    f1s  = np.array(f1s)
    aucs = np.array([a for a in aucs if not np.isnan(a)])

    print(f'\nResultados K-NN baseline (k={k_shot}, {n_repeats} repeticiones):')
    print(f'  Accuracy : {accs.mean():.4f} ± {accs.std(ddof=1):.4f}')
    print(f'  F1 macro : {f1s.mean():.4f} ± {f1s.std(ddof=1):.4f}')
    print(f'  ROC AUC  : {aucs.mean():.4f} ± {aucs.std(ddof=1):.4f}')

    return {'acc': float(accs.mean()), 'acc_std': float(accs.std(ddof=1)),
            'f1':  float(f1s.mean()),  'f1_std':  float(f1s.std(ddof=1)),
            'auc': float(aucs.mean()), 'auc_std': float(aucs.std(ddof=1))}


results = {}
for k in [1, 5, 10, 20]:
    print(f'\n{'='*50}')
    results[k] = run_knn_baseline(k_shot=k, n_repeats=100)

print('\n\n=== RESUMEN FINAL ===')
print(f'{"K-shot":>8} | {"Accuracy":>12} | {"F1 macro":>12} | {"ROC AUC":>12}')
print('-' * 52)
for k, r in results.items():
    print(f'{k:>8} | {r["acc"]:.4f}±{r["acc_std"]:.4f} | {r["f1"]:.4f}±{r["f1_std"]:.4f} | {r["auc"]:.4f}±{r["auc_std"]:.4f}')

with open(os.path.join(cfg.OUT_DIR, 'knn_baseline_results.json'), 'w') as f:
    json.dump(results, f, indent=2)


Test: 249 imgs, 5 clases
K-shot=1, n_repeats=100
Extrayendo embeddings de test...


Repeticiones: 100%|██████████| 100/100 [00:00<00:00, 210.36it/s]    



Resultados K-NN baseline (k=1, 100 repeticiones):
  Accuracy : 0.3281 ± 0.0699
  F1 macro : 0.3280 ± 0.0718
  ROC AUC  : 0.5910 ± 0.0459

Test: 249 imgs, 5 clases
K-shot=5, n_repeats=100
Extrayendo embeddings de test...


Repeticiones: 100%|██████████| 100/100 [00:00<00:00, 201.00it/s]    



Resultados K-NN baseline (k=5, 100 repeticiones):
  Accuracy : 0.3530 ± 0.0408
  F1 macro : 0.3648 ± 0.0454
  ROC AUC  : 0.6848 ± 0.0263

Test: 249 imgs, 5 clases
K-shot=10, n_repeats=100
Extrayendo embeddings de test...


Repeticiones: 100%|██████████| 100/100 [00:00<00:00, 192.78it/s]    



Resultados K-NN baseline (k=10, 100 repeticiones):
  Accuracy : 0.3774 ± 0.0289
  F1 macro : 0.3943 ± 0.0313
  ROC AUC  : 0.7220 ± 0.0205

Test: 249 imgs, 5 clases
K-shot=20, n_repeats=100
Extrayendo embeddings de test...


Repeticiones: 100%|██████████| 100/100 [00:00<00:00, 189.70it/s]    



Resultados K-NN baseline (k=20, 100 repeticiones):
  Accuracy : 0.3679 ± 0.0293
  F1 macro : 0.3914 ± 0.0277
  ROC AUC  : 0.7502 ± 0.0208


=== RESUMEN FINAL ===
  K-shot |     Accuracy |     F1 macro |      ROC AUC
----------------------------------------------------
       1 | 0.3281±0.0699 | 0.3280±0.0718 | 0.5910±0.0459
       5 | 0.3530±0.0408 | 0.3648±0.0454 | 0.6848±0.0263
      10 | 0.3774±0.0289 | 0.3943±0.0313 | 0.7220±0.0205
      20 | 0.3679±0.0293 | 0.3914±0.0277 | 0.7502±0.0208
